# Speculative Decoding: A100 Data Collection

**Project:** Adaptive Speculation Length Optimization for Speculative Decoding
**Courses:** MSML 604 (Optimization) + MSML 605 (Computing Systems)
**Hardware:** NVIDIA A100 (Google Colab Pro)
**Models:** Qwen 2.5 0.5B (draft) + Qwen 2.5 7B (target)

## What this notebook does

This is the main data collection notebook. With A100 access, we finally run the proper setup that speculative decoding was designed for (small draft + large target with ~14x parameter ratio).

**Sections:**
1. Environment setup and GPU check
2. Load models (0.5B + 7B)
3. Speculative decoding implementation with incremental KV cache
4. Sanity check (baseline vs speculative)
5. Rich data collection across k values and task types
6. Per-position acceptance analysis (for 604 IID assumption)
7. Save all results for downstream analysis

## 1. Environment Setup

**Runtime → Change runtime type → A100 GPU**.

In [ ]:
# Check GPU - should show A100
import subprocess
print(subprocess.check_output(['nvidia-smi']).decode())

In [ ]:
# Install dependencies (Colab has most preinstalled, just ensure versions)
!pip install -q transformers==4.44.2 accelerate
!pip install -q pynvml
print("Dependencies ready.")

In [ ]:
# Core imports
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
import time
import numpy as np
import json
import os
from dataclasses import dataclass, field
from typing import List, Optional, Tuple
from pathlib import Path

try:
    import pynvml
    pynvml.nvmlInit()
    nvml_handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    HAS_NVML = True
except Exception as e:
    print(f"NVML not available: {e}")
    HAS_NVML = False

torch.manual_seed(42)
np.random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Total memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Config
DRAFT_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
TARGET_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# Output directory - in Colab's local filesystem
OUTPUT_DIR = Path("/content/spec_decoding_results")
OUTPUT_DIR.mkdir(exist_ok=True)
print(f"\nResults will be saved to: {OUTPUT_DIR}")

### Optional: Mount Google Drive for persistent storage

In [ ]:
# OPTIONAL: Mount Google Drive (uncomment to use)
# from google.colab import drive
# drive.mount('/content/drive')
# OUTPUT_DIR = Path("/content/drive/MyDrive/spec_decoding_results")
# OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
# print(f"Results will be saved to Drive: {OUTPUT_DIR}")

## 2. Load Models

In [ ]:
# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(TARGET_MODEL_NAME)
print(f"Vocabulary size: {len(tokenizer)}")

# Load draft model
print("\nLoading draft model (Qwen 2.5 0.5B)...")
t0 = time.time()
draft_model = AutoModelForCausalLM.from_pretrained(
    DRAFT_MODEL_NAME,
    torch_dtype=torch.float16,
    device_map=device,
)
draft_model.eval()
print(f"Draft loaded in {time.time()-t0:.1f}s")
print(f"Draft parameters: {sum(p.numel() for p in draft_model.parameters()) / 1e6:.1f}M")

In [ ]:
# Load target model
print("Loading target model (Qwen 2.5 7B)...")
print("This will take 3-5 minutes on first run (downloading ~15GB)...")
t0 = time.time()
target_model = AutoModelForCausalLM.from_pretrained(
    TARGET_MODEL_NAME,
    torch_dtype=torch.float16,
    device_map=device,
)
target_model.eval()
print(f"Target loaded in {time.time()-t0:.1f}s")
print(f"Target parameters: {sum(p.numel() for p in target_model.parameters()) / 1e9:.2f}B")

assert draft_model.config.vocab_size == target_model.config.vocab_size
print(f"\nVocab sizes match: {draft_model.config.vocab_size}")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"GPU memory free:  {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.2f} GB")
print(f"\nDraft to target parameter ratio: 1 : {target_model.num_parameters() / draft_model.num_parameters():.1f}")

## 3. Instrumentation and Speculative Decoding

Our validated speculative decoding implementation with proper incremental KV cache management.

In [ ]:
@dataclass
class GenerationMetrics:
    total_time: float = 0.0
    prefill_time: float = 0.0
    decode_time: float = 0.0
    total_tokens_generated: int = 0
    num_decode_steps: int = 0
    step_times: List[float] = field(default_factory=list)
    k: Optional[int] = None
    tokens_proposed: int = 0
    tokens_accepted: int = 0
    acceptance_per_round: List[int] = field(default_factory=list)
    acceptance_per_position: List[List[int]] = field(default_factory=list)
    peak_memory_mb: float = 0.0
    avg_power_w: Optional[float] = None
    power_samples: List[float] = field(default_factory=list)
    
    @property
    def tokens_per_second(self) -> float:
        return self.total_tokens_generated / self.total_time if self.total_time > 0 else 0
    
    @property
    def overall_acceptance_rate(self) -> Optional[float]:
        return self.tokens_accepted / self.tokens_proposed if self.tokens_proposed > 0 else None
    
    def to_dict(self):
        return {
            'total_time': self.total_time,
            'prefill_time': self.prefill_time,
            'decode_time': self.decode_time,
            'total_tokens_generated': self.total_tokens_generated,
            'num_decode_steps': self.num_decode_steps,
            'step_times': self.step_times,
            'k': self.k,
            'tokens_proposed': self.tokens_proposed,
            'tokens_accepted': self.tokens_accepted,
            'acceptance_per_round': self.acceptance_per_round,
            'acceptance_per_position': self.acceptance_per_position,
            'peak_memory_mb': self.peak_memory_mb,
            'avg_power_w': self.avg_power_w,
            'tokens_per_second': self.tokens_per_second,
            'overall_acceptance_rate': self.overall_acceptance_rate,
        }
    
    def summary(self):
        print(f"Time: {self.total_time:.3f}s (prefill {self.prefill_time:.3f}s, decode {self.decode_time:.3f}s)")
        print(f"Tokens: {self.total_tokens_generated} in {self.num_decode_steps} steps")
        print(f"Throughput: {self.tokens_per_second:.1f} tokens/sec")
        if self.k is not None:
            print(f"k={self.k}, proposed {self.tokens_proposed}, accepted {self.tokens_accepted} ({self.overall_acceptance_rate:.2%})")
            print(f"Avg accepted per round: {np.mean(self.acceptance_per_round):.2f}")
        print(f"Peak GPU memory: {self.peak_memory_mb:.0f} MB")

def get_gpu_power_watts():
    if not HAS_NVML:
        return None
    try:
        return pynvml.nvmlDeviceGetPowerUsage(nvml_handle) / 1000.0
    except:
        return None

print("Instrumentation defined.")

In [ ]:
@torch.no_grad()
def baseline_generate(model, tokenizer, prompt, max_new_tokens=128, temperature=0.0, record_power=False):
    metrics = GenerationMetrics()
    torch.cuda.reset_peak_memory_stats()
    
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    input_ids = inputs.input_ids
    
    torch.cuda.synchronize()
    t_start = time.perf_counter()
    
    t_prefill = time.perf_counter()
    outputs = model(input_ids, use_cache=True)
    past_kv = outputs.past_key_values
    next_logits = outputs.logits[:, -1, :]
    torch.cuda.synchronize()
    metrics.prefill_time = time.perf_counter() - t_prefill
    
    if temperature == 0:
        next_token = torch.argmax(next_logits, dim=-1, keepdim=True)
    else:
        probs = F.softmax(next_logits / temperature, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
    
    generated_ids = [next_token.item()]
    
    t_decode = time.perf_counter()
    for _ in range(max_new_tokens - 1):
        step_start = time.perf_counter()
        if record_power:
            p = get_gpu_power_watts()
            if p: metrics.power_samples.append(p)
        
        outputs = model(next_token, past_key_values=past_kv, use_cache=True)
        past_kv = outputs.past_key_values
        next_logits = outputs.logits[:, -1, :]
        
        if temperature == 0:
            next_token = torch.argmax(next_logits, dim=-1, keepdim=True)
        else:
            probs = F.softmax(next_logits / temperature, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
        
        generated_ids.append(next_token.item())
        torch.cuda.synchronize()
        metrics.step_times.append(time.perf_counter() - step_start)
        metrics.num_decode_steps += 1
        
        if next_token.item() == tokenizer.eos_token_id:
            break
    
    torch.cuda.synchronize()
    metrics.decode_time = time.perf_counter() - t_decode
    metrics.total_time = time.perf_counter() - t_start
    metrics.total_tokens_generated = len(generated_ids)
    metrics.peak_memory_mb = torch.cuda.max_memory_allocated() / 1e6
    if metrics.power_samples:
        metrics.avg_power_w = float(np.mean(metrics.power_samples))
    
    return tokenizer.decode(generated_ids, skip_special_tokens=True), metrics


@torch.no_grad()
def _truncate_kv(past_key_values, target_length):
    truncated = []
    for layer_kv in past_key_values:
        k, v = layer_kv
        truncated.append((k[:, :, :target_length, :].contiguous(),
                         v[:, :, :target_length, :].contiguous()))
    return tuple(truncated)


@torch.no_grad()
def speculative_decode(draft_model, target_model, tokenizer, prompt, 
                        max_new_tokens=128, k=4, temperature=0.0, record_power=False):
    metrics = GenerationMetrics(k=k)
    torch.cuda.reset_peak_memory_stats()
    
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    prompt_ids = inputs.input_ids
    prompt_len = prompt_ids.shape[1]
    full_ids = prompt_ids.clone()
    
    torch.cuda.synchronize()
    t_start = time.perf_counter()
    
    t_prefill = time.perf_counter()
    target_out = target_model(prompt_ids, use_cache=True)
    target_kv = target_out.past_key_values
    next_target_logits = target_out.logits[:, -1, :]
    
    draft_out = draft_model(prompt_ids, use_cache=True)
    draft_kv = draft_out.past_key_values
    next_draft_logits = draft_out.logits[:, -1, :]
    torch.cuda.synchronize()
    metrics.prefill_time = time.perf_counter() - t_prefill
    
    t_decode = time.perf_counter()
    n_generated = 0
    
    while n_generated < max_new_tokens:
        round_start = time.perf_counter()
        if record_power:
            p = get_gpu_power_watts()
            if p: metrics.power_samples.append(p)
        
        # DRAFT
        draft_tokens = []
        draft_probs = []
        cur_logits = next_draft_logits
        cur_kv = draft_kv
        for i in range(k):
            if temperature == 0:
                probs = F.softmax(cur_logits, dim=-1)
                tok = torch.argmax(cur_logits, dim=-1, keepdim=True)
            else:
                probs = F.softmax(cur_logits / temperature, dim=-1)
                tok = torch.multinomial(probs, num_samples=1)
            draft_tokens.append(tok)
            draft_probs.append(probs)
            d_out = draft_model(tok, past_key_values=cur_kv, use_cache=True)
            cur_kv = d_out.past_key_values
            cur_logits = d_out.logits[:, -1, :]
        
        draft_seq = torch.cat(draft_tokens, dim=1)
        
        # VERIFY
        target_out = target_model(draft_seq, past_key_values=target_kv, use_cache=True)
        target_logits = target_out.logits
        target_kv_ext = target_out.past_key_values
        
        # REJECTION SAMPLING
        n_accepted = 0
        per_position_accept = []
        next_token = None
        
        for i in range(k):
            t_logits_here = next_target_logits if i == 0 else target_logits[:, i-1, :]
            
            if temperature == 0:
                t_top = torch.argmax(t_logits_here, dim=-1)
                if t_top.item() == draft_tokens[i].item():
                    n_accepted += 1
                    per_position_accept.append(1)
                else:
                    per_position_accept.append(0)
                    next_token = t_top.unsqueeze(0)
                    break
            else:
                p_t = F.softmax(t_logits_here / temperature, dim=-1)
                p_d = draft_probs[i]
                tid = draft_tokens[i].item()
                accept_prob = min(1.0, p_t[0, tid].item() / max(p_d[0, tid].item(), 1e-12))
                if torch.rand(1).item() < accept_prob:
                    n_accepted += 1
                    per_position_accept.append(1)
                else:
                    per_position_accept.append(0)
                    adjusted = torch.clamp(p_t - p_d, min=0)
                    adjusted = adjusted / (adjusted.sum() + 1e-12)
                    next_token = torch.multinomial(adjusted, num_samples=1)
                    break
        
        if n_accepted == k:
            bonus_logits = target_logits[:, k-1, :]
            if temperature == 0:
                next_token = torch.argmax(bonus_logits, dim=-1, keepdim=True)
            else:
                bp = F.softmax(bonus_logits / temperature, dim=-1)
                next_token = torch.multinomial(bp, num_samples=1)
        
        # COMMIT
        for i in range(n_accepted):
            full_ids = torch.cat([full_ids, draft_tokens[i]], dim=1)
        full_ids = torch.cat([full_ids, next_token], dim=1)
        n_generated += n_accepted + 1
        
        metrics.tokens_proposed += k
        metrics.tokens_accepted += n_accepted
        metrics.acceptance_per_round.append(n_accepted)
        metrics.acceptance_per_position.append(per_position_accept + [-1] * (k - len(per_position_accept)))
        metrics.num_decode_steps += 1
        
        if next_token.item() == tokenizer.eos_token_id or n_generated >= max_new_tokens:
            torch.cuda.synchronize()
            metrics.step_times.append(time.perf_counter() - round_start)
            break
        
        # UPDATE KV CACHES
        cur_target_kv_len = target_kv_ext[0][0].shape[2]
        accepted_prefix_len = cur_target_kv_len - k + n_accepted
        target_kv = _truncate_kv(target_kv_ext, accepted_prefix_len)
        t_out = target_model(next_token, past_key_values=target_kv, use_cache=True)
        target_kv = t_out.past_key_values
        next_target_logits = t_out.logits[:, -1, :]
        
        cur_draft_kv_len = cur_kv[0][0].shape[2]
        accepted_prefix_len_d = cur_draft_kv_len - k + n_accepted
        draft_kv = _truncate_kv(cur_kv, accepted_prefix_len_d)
        d_out = draft_model(next_token, past_key_values=draft_kv, use_cache=True)
        draft_kv = d_out.past_key_values
        next_draft_logits = d_out.logits[:, -1, :]
        
        torch.cuda.synchronize()
        metrics.step_times.append(time.perf_counter() - round_start)
    
    torch.cuda.synchronize()
    metrics.decode_time = time.perf_counter() - t_decode
    metrics.total_time = time.perf_counter() - t_start
    metrics.total_tokens_generated = n_generated
    metrics.peak_memory_mb = torch.cuda.max_memory_allocated() / 1e6
    if metrics.power_samples:
        metrics.avg_power_w = float(np.mean(metrics.power_samples))
    
    generated_ids = full_ids[0, prompt_len:].tolist()
    return tokenizer.decode(generated_ids, skip_special_tokens=True), metrics


print("Decoding functions defined.")

## 4. Sanity Check

Verify:
1. Baseline and speculative outputs match (correctness)
2. Speculative gives actual speedup on A100 + 7B

In [ ]:
# Warm up GPU
print("Warming up GPU...")
_, _ = baseline_generate(target_model, tokenizer, "Hello world", max_new_tokens=10)
print("Warmup done.\n")

test_prompt = "Explain the key differences between supervised and unsupervised learning in machine learning."

print("=" * 70)
print("BASELINE (target model autoregressive)")
print("=" * 70)
text_base, m_base = baseline_generate(target_model, tokenizer, test_prompt, max_new_tokens=80)
print(f"Output: {text_base}\n")
m_base.summary()

print("\n" + "=" * 70)
print("SPECULATIVE (k=4, greedy)")
print("=" * 70)
text_spec, m_spec = speculative_decode(draft_model, target_model, tokenizer, test_prompt,
                                         max_new_tokens=80, k=4, temperature=0)
print(f"Output: {text_spec}\n")
m_spec.summary()

print("\n" + "=" * 70)
print("COMPARISON")
print("=" * 70)
print(f"Baseline:    {m_base.tokens_per_second:.1f} tokens/sec")
print(f"Speculative: {m_spec.tokens_per_second:.1f} tokens/sec")
speedup = m_spec.tokens_per_second / m_base.tokens_per_second
print(f"Speedup: {speedup:.2f}x")

matches = text_spec[:40] == text_base[:40]
print(f"\nOutputs match for first 40 chars: {matches}")
if matches and speedup > 1.0:
    print("\nBOTH CORRECTNESS AND PERFORMANCE CHECKS PASSED.")
    print(f"Speculative decoding is {speedup:.2f}x faster on A100 with 7B target.")
elif matches:
    print("\nCorrect, but speedup is less than 1x.")
else:
    print("\nCorrectness issue. Outputs diverge.")

## 5. Rich Data Collection

Collects speculative decoding data across:
- Multiple prompts covering different task types
- k values from 1 to 10
- Multiple runs per config for statistical error bars

In [ ]:
prompts_by_category = {
    'factual_qa': [
        "What are the three branches of the US government?",
        "When did World War II end?",
        "What is the capital city of Australia?",
        "Who wrote the novel 1984?",
        "What is the speed of light in meters per second?",
    ],
    'code': [
        "Write a Python function that computes the Fibonacci sequence up to n:",
        "Implement binary search in Python:",
        "Write a SQL query to find the top 5 highest-paid employees:",
        "Create a Python class representing a bank account with deposit and withdraw methods:",
        "Write a JavaScript function to check if a string is a palindrome:",
    ],
    'summarization': [
        "Summarize the key ideas of supervised learning in 3 sentences:",
        "In a paragraph, explain what a transformer model is:",
        "Describe the water cycle in simple terms:",
        "Summarize how photosynthesis works:",
        "Give a brief overview of how the internet works:",
    ],
    'reasoning': [
        "If a train leaves station A at 2pm traveling 60 mph and station B is 180 miles away, when does it arrive?",
        "A farmer has 17 sheep. All but 9 die. How many are left?",
        "If you double a number and then add 10, you get 34. What is the number?",
        "Why do objects of different masses fall at the same rate in a vacuum?",
        "Explain why ice floats on water:",
    ],
    'conversational': [
        "What do you think about remote work becoming the new normal?",
        "I am feeling overwhelmed with my studies. Any advice?",
        "Tell me an interesting fact about octopuses:",
        "What are some good habits for staying productive?",
        "What makes a good friend?",
    ],
}

all_prompts = []
for cat, ps in prompts_by_category.items():
    for p in ps:
        all_prompts.append({'category': cat, 'prompt': p})

print(f"Total prompts: {len(all_prompts)} across {len(prompts_by_category)} categories")
for cat, ps in prompts_by_category.items():
    print(f"  {cat}: {len(ps)}")

In [ ]:
# Run the big experiment
k_values = [1, 2, 3, 4, 5, 6, 8, 10]
MAX_NEW_TOKENS = 100
NUM_RUNS_PER_CONFIG = 2

all_results = {
    'metadata': {
        'draft_model': DRAFT_MODEL_NAME,
        'target_model': TARGET_MODEL_NAME,
        'gpu': torch.cuda.get_device_name(0),
        'max_new_tokens': MAX_NEW_TOKENS,
        'num_runs_per_config': NUM_RUNS_PER_CONFIG,
    },
    'baseline': [],
    'speculative': []
}

print("Collecting baseline data (autoregressive, target model)...")
for i, p_info in enumerate(all_prompts):
    if i % 5 == 0:
        print(f"  Baseline progress: {i}/{len(all_prompts)}")
    text, m = baseline_generate(target_model, tokenizer, p_info['prompt'], 
                                  max_new_tokens=MAX_NEW_TOKENS, temperature=0)
    all_results['baseline'].append({
        'category': p_info['category'],
        'prompt': p_info['prompt'],
        'output': text,
        'metrics': m.to_dict(),
    })

print(f"\nBaseline done. Mean throughput: {np.mean([r['metrics']['tokens_per_second'] for r in all_results['baseline']]):.1f} tok/s")

In [ ]:
# Speculative sweep
print("Collecting speculative data across k values...")
total_configs = len(k_values) * len(all_prompts) * NUM_RUNS_PER_CONFIG
done = 0

for k in k_values:
    print(f"\n=== k = {k} ===")
    for p_info in all_prompts:
        for run_idx in range(NUM_RUNS_PER_CONFIG):
            text, m = speculative_decode(draft_model, target_model, tokenizer, 
                                           p_info['prompt'],
                                           max_new_tokens=MAX_NEW_TOKENS, 
                                           k=k, temperature=0)
            all_results['speculative'].append({
                'k': k,
                'run_idx': run_idx,
                'category': p_info['category'],
                'prompt': p_info['prompt'],
                'output': text,
                'metrics': m.to_dict(),
            })
            done += 1
    
    k_results = [r for r in all_results['speculative'] if r['k'] == k]
    mean_tps = np.mean([r['metrics']['tokens_per_second'] for r in k_results])
    mean_accept = np.mean([r['metrics']['overall_acceptance_rate'] for r in k_results])
    mean_accepted_per_round = np.mean([np.mean(r['metrics']['acceptance_per_round']) for r in k_results])
    print(f"  Completed {len(k_results)} runs. Mean throughput: {mean_tps:.1f} tok/s, "
          f"acceptance: {mean_accept:.3f}, avg accepted/round: {mean_accepted_per_round:.2f}")
    print(f"  Overall progress: {done}/{total_configs}")

print("\nData collection complete!")

## 6. Quick Analysis

In [ ]:
# Summary table
baseline_throughput = np.mean([r['metrics']['tokens_per_second'] for r in all_results['baseline']])
print(f"Baseline throughput (autoregressive target): {baseline_throughput:.1f} tokens/sec\n")

print(f"{'k':>3} | {'Throughput':>12} | {'Speedup':>8} | {'AcceptRate':>11} | {'Accept/Round':>12}")
print("-" * 60)
for k in k_values:
    k_runs = [r for r in all_results['speculative'] if r['k'] == k]
    mean_tps = np.mean([r['metrics']['tokens_per_second'] for r in k_runs])
    mean_accept = np.mean([r['metrics']['overall_acceptance_rate'] for r in k_runs])
    mean_accepted_per_round = np.mean([np.mean(r['metrics']['acceptance_per_round']) for r in k_runs])
    speedup = mean_tps / baseline_throughput
    print(f"{k:>3} | {mean_tps:>10.1f} t/s | {speedup:>6.2f}x | {mean_accept:>11.3f} | {mean_accepted_per_round:>12.2f}")

In [ ]:
# Throughput by category and k
print("Throughput by category and k:\n")
categories = list(prompts_by_category.keys())
print(f"{'Category':<18} | " + " | ".join(f"k={k}" for k in k_values))
print("-" * (22 + len(k_values) * 6))
for cat in categories:
    row = []
    for k in k_values:
        runs = [r for r in all_results['speculative'] if r['k'] == k and r['category'] == cat]
        tps = np.mean([r['metrics']['tokens_per_second'] for r in runs])
        row.append(f"{tps:.1f}")
    print(f"{cat:<18} | " + " | ".join(f"{v:>4}" for v in row))

## 7. Per-Position Acceptance (IID Assumption Check)

Whether per-token acceptance can be assumed IID? Check empirically.

- If IID holds: acceptance rate is constant across positions within a round
- If not IID: acceptance rate typically drops as position increases

In [ ]:
# Per-position acceptance rate
print("Per-position acceptance rate (IID assumption check):\n")
print(f"{'Position':>9} | " + " | ".join(f"k={k}" for k in k_values if k >= 4))
print("-" * 60)

for pos in range(10):
    row = []
    for k in k_values:
        if k < pos + 1:
            row.append("  -  ")
            continue
        rounds_with_this_pos = []
        for r in all_results['speculative']:
            if r['k'] != k: continue
            for round_data in r['metrics']['acceptance_per_position']:
                if pos < len(round_data) and round_data[pos] != -1:
                    rounds_with_this_pos.append(round_data[pos])
        if rounds_with_this_pos:
            rate = np.mean(rounds_with_this_pos)
            row.append(f"{rate:.3f}")
        else:
            row.append("  -  ")
    if any(v != "  -  " for v in row):
        print(f"pos {pos:>3}  | " + " | ".join(f"{v:>5}" for v in row))

print("\nIf numbers DROP across positions, IID assumption is violated.")
print("If they stay roughly constant, IID is a reasonable first approximation.")

## 8. Save All Data

In [ ]:
# Save to output directory
output_file = OUTPUT_DIR / "rich_data_a100.json"
with open(output_file, 'w') as f:
    json.dump(all_results, f, indent=2, default=str)

print(f"Data saved to: {output_file}")
print(f"File size: {output_file.stat().st_size / 1e6:.2f} MB")
print(f"\nTotal baseline runs:    {len(all_results['baseline'])}")
print(f"Total speculative runs: {len(all_results['speculative'])}")

summary = {
    'metadata': all_results['metadata'],
    'baseline_throughput_mean': float(np.mean([r['metrics']['tokens_per_second'] for r in all_results['baseline']])),
    'baseline_throughput_std': float(np.std([r['metrics']['tokens_per_second'] for r in all_results['baseline']])),
    'k_sweep': {}
}
for k in k_values:
    k_runs = [r for r in all_results['speculative'] if r['k'] == k]
    summary['k_sweep'][str(k)] = {
        'throughput_mean': float(np.mean([r['metrics']['tokens_per_second'] for r in k_runs])),
        'throughput_std': float(np.std([r['metrics']['tokens_per_second'] for r in k_runs])),
        'acceptance_rate_mean': float(np.mean([r['metrics']['overall_acceptance_rate'] for r in k_runs])),
        'accepted_per_round_mean': float(np.mean([np.mean(r['metrics']['acceptance_per_round']) for r in k_runs])),
        'speedup': float(np.mean([r['metrics']['tokens_per_second'] for r in k_runs]) / summary['baseline_throughput_mean']),
    }

with open(OUTPUT_DIR / "summary_a100.json", 'w') as f:
    json.dump(summary, f, indent=2)
print(f"Summary saved to: {OUTPUT_DIR / 'summary_a100.json'}")

## Download the results

In [ ]:
from google.colab import files
files.download(str(OUTPUT_DIR / "rich_data_a100.json"))
files.download(str(OUTPUT_DIR / "summary_a100.json"))

**For MSML 604:**
- Latency data across k values → fit the cost model
- Acceptance rates and per-position acceptance → IID analysis  
- Multiple task categories → show domain dependence of optimal k

**For MSML 605:**
- Baseline throughput for the 7B target
- Speculative throughput across k values → characterize the speedup curve